### **1) Import Modules**

In [1]:
# torch handles tensors, gradients, and models. numpy handles arrays.
import torch
import numpy as np
# pyplot is used later to draw the sine curve and loss plots.
import matplotlib.pyplot as plt

### **2) Tensors**

In PyTorch, tensors are multidimensional arrays, similar to NumPy arrays, that support automatic differentiation and can be processed on GPUs for efficient computation. Creating tensors and performing operations on them is very similar to working with NumPy arrays, with many functions sharing the same names and syntax. PyTorch supports familiar operations such as indexing, slicing, reshaping, element-wise arithmetic, and matrix multiplication.

In [2]:
# Scalar (0D tensor)
# One value with no axes, so its shape is ().
a = torch.tensor(5)

# Vector (1D tensor)
# One axis with 3 values. The shape is (3,).
b = torch.tensor([1, 2, 3])

# Matrix (2D tensor)
# Read (2, 3) as 2 rows with 3 values per row.
c = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

# 3D tensor
# Read (2, 2, 2) from the outside in: blocks, rows per block, values per row.
d = torch.tensor([[[1, 2], [3, 4]],
                  [[5, 6], [7, 8]]])

# Count the numbers in the shape to get the number of dimensions.
print(type(a))
print(a.shape)  # torch.Size([])
print(b.shape)  # torch.Size([3])
print(c.shape)  # torch.Size([2, 3])
print(d.shape)  # torch.Size([2, 2, 2])

<class 'torch.Tensor'>
torch.Size([])
torch.Size([3])
torch.Size([2, 3])
torch.Size([2, 2, 2])


In [3]:
# creating tensors of arbitrary shapes

a = torch.zeros(3, 4)       # 3x4 tensor of zeros
b = torch.ones(2, 3, 4)     # 2x3x4 tensor of ones
# rand picks uniformly from 0 up to 1, excluding 1.
c = torch.rand(2, 3)        # 2x3 tensor of random values
# randn uses a bell curve centered at 0 with standard deviation 1.
# Negative values and values outside -1 to 1 are possible.
d = torch.randn(3, 4)       # 3x4 tensor from standard normal distribution

print(a)
print(b)
print(c)
print(d)

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])
tensor([[0.6683, 0.2312, 0.2192],
        [0.6091, 0.8053, 0.6161]])
tensor([[-0.8138,  0.2532, -0.2435, -0.6569],
        [-0.3194,  1.1854, -1.0383, -0.8275],
        [-0.9553,  0.7577,  0.5980, -0.1892]])


In [4]:
# tensor operations and indexing

# a and b each have shape (2, 2).
a = torch.tensor([[1., 2.],
                  [3., 4.]])

b = torch.tensor([[5., 6.],
                  [7., 8.]])

# Element-wise operations
# Pair values in the same positions. For a * b, the top left is 1 * 5 = 5.
print(a + b)       # Addition
print(a - b)       # Subtraction
print(a * b)       # Multiplication
print(a / b)       # Division
# ** 2 squares each value and keeps the shape.
print(a ** 2)      # Exponentiation

# Matrix operations
# @ takes a row from a and a column from b. Top left: 1*5 + 2*7 = 19.
print(a @ b)       # Matrix multiplication
# Transpose swaps rows and columns. The first row [1, 2] becomes the first column.
print(a.T)         # Transpose

# Aggregation operations
# With no dimension specified, each operation reduces all four values to one scalar.
print(a.sum())     # Sum of all elements
print(a.mean())    # Mean of all elements
print(a.max())     # Maximum element

# Reshaping
# Both shapes keep the same four values in order. (4,) is 1D, (1, 4) is 2D.
print(a.reshape(4))    # Flatten into 1D tensor
print(a.reshape(1, 4)) # Reshape into 1x4 tensor

# Indexing and slicing
# Read a[row, column]. Indices start at 0. Two integer indices give a scalar.
print(a[0, 1])     # Element in row 0, column 1
print(a[0])        # First row
# : keeps all rows. Column index 1 gives [2, 4] with shape (2,).
# A slice keeps its axis: a[:, 1:] gives [[2], [4]] with shape (2, 1).
# Slice format is start:stop:step. The stop index is excluded.
print(a[:, 1])     # Second column

# Element-wise functions
# Each function acts on one entry at a time and keeps the shape (2, 2).
print(torch.exp(a))
print(torch.log(a))
print(torch.sqrt(a))
# Sigmoid maps values between 0 and 1. A zero input maps to 0.5.
print(torch.sigmoid(a))
# ReLU replaces negative values with 0. Every value in a is positive.
print(torch.relu(a))

tensor([[ 6.,  8.],
        [10., 12.]])
tensor([[-4., -4.],
        [-4., -4.]])
tensor([[ 5., 12.],
        [21., 32.]])
tensor([[0.2000, 0.3333],
        [0.4286, 0.5000]])
tensor([[ 1.,  4.],
        [ 9., 16.]])
tensor([[19., 22.],
        [43., 50.]])
tensor([[1., 3.],
        [2., 4.]])
tensor(10.)
tensor(2.5000)
tensor(4.)
tensor([1., 2., 3., 4.])
tensor([[1., 2., 3., 4.]])
tensor(2.)
tensor([1., 2.])
tensor([2., 4.])
tensor([[ 2.7183,  7.3891],
        [20.0855, 54.5981]])
tensor([[0.0000, 0.6931],
        [1.0986, 1.3863]])
tensor([[1.0000, 1.4142],
        [1.7321, 2.0000]])
tensor([[0.7311, 0.8808],
        [0.9526, 0.9820]])
tensor([[1., 2.],
        [3., 4.]])


### **3) Computing Gradients of Tensors**

One of the most important features of PyTorch is its ability to automatically compute gradients using automatic differentiation. By setting requires_grad=True when creating a tensor, PyTorch tracks operations involving that tensor and allows us to compute gradients using the .backward() method.

In [5]:
# Create a tensor that tracks gradients
# requires_grad=True lets PyTorch track how y is computed from x.
x = torch.tensor(3.0, requires_grad=True)

# Define a function: y = x^2 + 2x + 1
# Forward pass: x = 3 gives y = 9 + 6 + 1 = 16.
y = x**2 + 2*x + 1

# Compute the gradient dy/dx
# Backward pass: compute the slope dy/dx and store it in x.grad.
y.backward()

# Access the gradient
# The slope is 2*x + 2 = 8 at x = 3. The value of x stays 3.
print(x.grad)  # tensor(8.)

tensor(8.)


In [6]:
# Create tensors
# x is the input and y is the true label. w and b are the model parameters.
x = torch.tensor(2.0)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(0.0)

# Forward pass
# Compute a score, then use sigmoid to get a prediction between 0 and 1.
z = w*x + b
y_hat = torch.sigmoid(z)

# Compute binary cross-entropy loss
# Binary cross-entropy penalizes a confident prediction of 1 when the label is 0.
loss = -(y*torch.log(y_hat) + (1-y)*torch.log(1-y_hat))

# Retain intermediate gradients
# Retain these intermediate gradients so we can print z.grad and y_hat.grad.
z.retain_grad()
y_hat.retain_grad()

# Backward pass
# Compute how sensitive the loss is to w and b. The results go in .grad.
loss.backward()

# Print results
# .item() extracts a Python number from a single-element tensor.
print("Prediction:", y_hat.item())
print("Loss:", loss.item())
print("Gradient of w:", w.grad)
print("Gradient of b:", b.grad)

# Print gradients
print("dL/dw:", w.grad)
print("dL/db:", b.grad)
print("dL/dz:", z.grad)
print("dL/dy_hat:", y_hat.grad)

Prediction: 0.9990890026092529
Loss: 7.000970363616943
Gradient of w: tensor(1.9982)
Gradient of b: tensor(0.9991)
dL/dw: tensor(1.9982)
dL/db: tensor(0.9991)
dL/dz: tensor(0.9991)
dL/dy_hat: tensor(1097.6980)


In [7]:
# multiple inputs

# Create tensors
# Three observations with one feature each. The same scalar w and b apply to each.
x = torch.tensor([2.0, 4.0, 1.0])
w = torch.tensor(0.3, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = torch.tensor([0.0, 1.0, 0.0])

# Forward pass
# Broadcast w and b across x. This gives three scores and three predictions.
z = w*x + b
y_hat = torch.sigmoid(z)

# Compute binary cross-entropy loss
# mean() averages the three per-observation losses into one scalar.
loss = -(y*torch.log(y_hat) + (1-y)*torch.log(1-y_hat)).mean()

# Backward pass
# The shared weight and bias each get one gradient from the averaged loss.
loss.backward()

# Print results
print("Predictions:", y_hat)
print("Loss:", loss.item())
print("Gradient of w:", w.grad)
print("Gradient of b:", b.grad)

Predictions: tensor([0.8320, 0.9002, 0.7858], grad_fn=<SigmoidBackward0>)
Loss: 1.1433309316635132
Gradient of w: tensor(0.6836)
Gradient of b: tensor(0.5060)


In [8]:
# vector of parameters

# Create tensors
# One observation with three features. Each feature has its own weight.
x = torch.tensor([2.0, 4.0, 1.0])
w = torch.tensor([3.0, 2.0, 1.0], requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(0.0)

# Forward pass
# Dot product: multiply matching entries and add them. Then add b to get one score.
# Here the score is 3*2 + 2*4 + 1*1 + 1 = 16, so sigmoid gives a value close to 1.
z = torch.dot(w, x) + b
y_hat = torch.sigmoid(z)

# Compute binary cross-entropy loss
# The target is 0, so that confident prediction produces a large loss.
loss = -(y*torch.log(y_hat) + (1-y)*torch.log(1-y_hat))


# Backward pass
# The backward pass computes gradients. w.grad has 3 values and b.grad has 1.
# Updating the weights would be a separate step.
loss.backward()

# Print results
print("Predictions:", y_hat)
print("Loss:", loss.item())
print("Gradient of w:", w.grad)
print("Gradient of b:", b.grad)

Predictions: tensor(1.0000, grad_fn=<SigmoidBackward0>)
Loss: 15.942384719848633
Gradient of w: tensor([2.0000, 4.0000, 1.0000])
Gradient of b: tensor(1.0000)


### **4) Generating Noisy Sine Regression Data**

Generate 1,200 observations from $y=\sin(x)+\epsilon$. Randomly split these observations into 70% training, 15% validation, and 15% test sets.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Use a seed so the sequence of random draws can be repeated under the same conditions.
torch.manual_seed(42)
# Choose GPU if CUDA is available. Tensors move when we call .to(device) later.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Generate 1,200 noisy observations from the same distribution.
num_samples = 1200
# Scale uniform draws to [-3, 3). X has one column for one feature.
X = -3 + 6 * torch.rand(num_samples, 1)
# Add normal noise with standard deviation 0.35. y has the same shape as X.
y = torch.sin(X) + 0.35 * torch.randn_like(X)

# Random 70% / 15% / 15% split (no overlap between sets).
indices = torch.randperm(num_samples)
train_end = int(0.70 * num_samples)
val_end = int(0.85 * num_samples)

# Slices of one shuffled index list give 840 train, 180 validation, and 180 test rows.
train_idx = indices[:train_end]
val_idx = indices[train_end:val_end]
test_idx = indices[val_end:]

# Use the same indices for X and y so every input stays paired with its target.
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)


### **5) Visualizing the Data**



In [ ]:
# Start at -3, end at 3, and create 500 evenly spaced values in one column.
x_plot = torch.linspace(-3, 3, 500).reshape(-1, 1)
plt.figure(figsize=(9, 5))
# The dashed line shows the clean sine signal. The dots show noisy observations.
plt.plot(x_plot.numpy(), torch.sin(x_plot).numpy(), 'k--', label='True sine curve')
plt.scatter(X_train.numpy(), y_train.numpy(), s=12, alpha=0.45, label='Training')
plt.scatter(X_val.numpy(), y_val.numpy(), s=12, alpha=0.45, label='Validation')
plt.scatter(X_test.numpy(), y_test.numpy(), s=12, alpha=0.45, label='Test')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()


### **6) Creating a Training DataLoader**

The `DataLoader` shuffles training observations and creates mini-batches. Validation and test sets can be evaluated in a single forward pass. Training batches are moved to the selected device inside the training loop.


In [ ]:
# TensorDataset pairs each input with its target. DataLoader shuffles the pairs.
# One epoch uses 26 full batches of 32 plus a last batch of 8.
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)


### **7) Defining an MLP**

This model has one input feature, three hidden layers of 128 neurons with ReLU activations, and one **linear** regression output.


In [ ]:
class SineMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Sequential runs these layers in order. Each hidden layer outputs (batch size, 128).
        self.network = nn.Sequential(
            nn.Linear(1, 128),
            # ReLU lets the network learn nonlinear patterns like a sine curve.
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            # One output per input row. The linear output can predict negative values.
            nn.Linear(128, 1)
        )

    def forward(self, x):
        # Calling model(x) runs this forward computation.
        return self.network(x)

# Move the model to the selected device before training.
model = SineMLP().to(device)
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))

### **8) Specifying the Loss Function and Optimizer**

Mean squared error (MSE) measures regression error; Adam updates the weights based on gradients computed by PyTorch.

In [ ]:
# Mean squared error averages the squared gaps between predictions and targets.
loss_fn = nn.MSELoss()
# Adam uses the gradients to update the parameters when optimizer.step() runs.
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

### **9) Training and Validation**

For each training batch: predict, compute loss, clear gradients, backpropagate, and update weights. After each epoch, measure validation MSE without computing gradients. Save the model parameters from the epoch with the lowest validation loss.


In [ ]:
# An epoch is one full pass through the training batches.
num_epochs = 600
train_losses = []
val_losses = []
# Start at infinity so the first finite validation loss becomes the best.
best_val_loss = float('inf')
best_epoch = 0
best_state = None

# Move validation data once, since it is evaluated each epoch.
X_val_device = X_val.to(device)
y_val_device = y_val.to(device)

# Wait for earlier GPU work before starting the timer.
if device.type == 'cuda':
    torch.cuda.synchronize()
start_time = time.perf_counter()

for epoch in range(num_epochs):
    model.train()
    train_loss_sum = 0.0

    for xb, yb in train_loader:
        # Move the batch to the same device as the model.
        xb, yb = xb.to(device), yb.to(device)
        # Forward pass: predict for this batch, then measure its error.
        predictions = model(xb)
        loss = loss_fn(predictions, yb)
        # Clear old gradients so they do not accumulate across batches.
        optimizer.zero_grad()
        # Compute gradients of this batch's loss.
        loss.backward()
        # Use those gradients to update the weights and biases.
        optimizer.step()
        # Multiply the batch mean by its size so the last small batch is weighted fairly.
        train_loss_sum += loss.item() * len(xb)

    train_losses.append(train_loss_sum / len(X_train))

    # Switch to evaluation mode. no_grad separately turns off gradient tracking.
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val_device), y_val_device).item()
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        # Clone the saved values so later updates cannot change this checkpoint.
        best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:4d}: train MSE={train_losses[-1]:.4f}, val MSE={val_losses[-1]:.4f}')

elapsed = time.perf_counter() - start_time
print(f'Training and validation time: {elapsed:.2f} seconds')
print(f'Lowest validation MSE: {best_val_loss:.4f} (epoch {best_epoch})')

### **10) Plotting Training and Validation Loss Curves**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training MSE')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation MSE')
# Mark the epoch with the lowest validation MSE.
# Falling train MSE with rising validation MSE can be a sign of overfitting.
plt.axvline(best_epoch, color='gray', linestyle='--', label=f'Best validation epoch ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### **11) Evaluating on the Test Set**

Compare the final model with the checkpoint that achieved the lowest validation MSE.


In [ ]:
# Evaluate the final epoch's model on the held-out test set.
model.eval()
X_test_device = X_test.to(device)
y_test_device = y_test.to(device)
with torch.no_grad():
    final_test_mse = loss_fn(model(X_test_device), y_test_device).item()

# Load the checkpoint chosen using validation loss, then score it on the same test set.
best_model = SineMLP().to(device)
best_model.load_state_dict(best_state)
best_model.eval()
with torch.no_grad():
    best_test_mse = loss_fn(best_model(X_test_device), y_test_device).item()

print(f'Final model test MSE: {final_test_mse:.4f}')
print(f'Best-validation model test MSE: {best_test_mse:.4f}')

### **12) Visualizing the Learned Curve**

Compare the final model, the best-validation model, the underlying sine curve , and the training points.


In [ ]:
# Use 500 evenly spaced x values in one feature column to draw smooth curves.
x_grid = torch.linspace(-3, 3, 500).reshape(-1, 1)
model.eval()
best_model.eval()
with torch.no_grad():
    # Move predictions to CPU before converting them to NumPy arrays for plotting.
    final_predictions = model(x_grid.to(device)).cpu().numpy()
    best_predictions = best_model(x_grid.to(device)).cpu().numpy()

plt.figure(figsize=(9, 5))
# Compare the two fitted curves with the clean sine curve and the noisy training points.
plt.scatter(X_train.numpy(), y_train.numpy(), s=12, alpha=0.4, label='Noisy training points')
plt.plot(x_grid.numpy(), torch.sin(x_grid).numpy(), 'k--', label='True sine curve')
plt.plot(x_grid.numpy(), best_predictions, label='Best-validation MLP')
plt.plot(x_grid.numpy(), final_predictions, label='Final MLP')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Sine Regression: Fitting the Signal vs. the Noise')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
